# OSC Grasp Training (Colab)

Runs `train_osc_grasp_bc_parallel.py` from the `Arm-OSC-Grasp` repo on a Colab runtime instead of the local machine — a behavior-cloning-warm-started variant of pure SAC, after 585k/1M steps of pure RL never once found a successful grasp despite the task being provably achievable (see `IMP_NOTES.md`). Cell 5.5 collects real successful demonstrations via a hand-scripted routine first; cell 6 seeds the replay buffer and pretrains the actor from them before RL fine-tuning begins.

**Runtime type**: a plain CPU runtime is fine — no need to select a GPU. This workload is CPU-bound MuJoCo physics plus a tiny MLP; `device="cpu"` is set explicitly in the training script regardless, and a T4 wouldn't speed this up (see `IMP_NOTES.md`).

**Why Drive is mounted**: Colab's local disk is wiped whenever the runtime disconnects or recycles (idle timeout, 12h session cap, etc.) — a multi-hour training run WILL eventually hit this. Checkpoints are symlinked into Google Drive below so they survive a disconnect; only re-run cells 1-4 to resume watching a run, and cells 5.5-6 again to re-collect demonstrations and continue/restart training.

In [1]:
# Cell 1 — mount Google Drive (checkpoints and the final model save both
# land here, not on Colab's ephemeral local disk)
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# Cell 2 — clone the repo. Leave the token prompt blank if the repo is public;
# paste a GitHub Personal Access Token (repo scope) if it's private.
import getpass, os

REPO_URL = "https://github.com/kaustubhadhe1206/Arm-OSC-Grasp.git"
REPO_DIR = "Arm-OSC-Grasp"

token = getpass.getpass("GitHub token (leave blank if repo is public): ")
clone_url = REPO_URL.replace("https://", f"https://{token}@") if token else REPO_URL

if os.path.isdir(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone {clone_url}

%cd {REPO_DIR}

Cloning into 'Arm-OSC-Grasp'...
remote: Enumerating objects: 117, done.
remote: Counting objects: 100% (117/117), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 117 (delta 19), reused 111 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (117/117), 4.87 MiB | 8.98 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/Arm-OSC-Grasp


In [3]:
# Cell 3 — point checkpoints at Drive via a symlink, so the training script's
# existing `save_path="./checkpoints/"` transparently writes to Drive instead
# of Colab's local (ephemeral) disk, with no changes needed to the script
# itself — keeps this notebook and the local project in sync.
import os

DRIVE_CKPT_DIR = "/content/drive/MyDrive/Arm-OSC-Grasp-checkpoints"
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)

if os.path.islink("checkpoints") or os.path.isdir("checkpoints"):
    !rm -rf checkpoints
!ln -s {DRIVE_CKPT_DIR} checkpoints

print("Checkpoints will be saved to:", DRIVE_CKPT_DIR)

Checkpoints will be saved to: /content/drive/MyDrive/Arm-OSC-Grasp-checkpoints


In [4]:
# Cell 4 — install dependencies. torch is already preinstalled on Colab (CUDA
# build) — that's fine, the training script forces device="cpu" regardless.
!pip install -q mujoco gymnasium stable-baselines3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.7/232.7 kB 2.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.8/19.8 MB 55.4 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.5/248.5 kB 26.7 MB/s eta 0:00:00


In [5]:
# Cell 5 — sanity check: how many CPU cores does this runtime actually have?
# Colab's free tier is often ~2 vCPUs — if this shows fewer than 4, consider
# lowering N_ENVS in train_osc_grasp_parallel.py (currently 4) to match,
# since oversubscribing cores for SubprocVecEnv workers can be SLOWER than
# fewer, uncontended workers.
import os
print("CPU count:", os.cpu_count())

CPU count: 2


In [ ]:
# Cell 5.5 — collect grasp demonstrations via a hand-scripted (no RL)
# routine, used to warm-start training below. Pure RL alone ran 585k/1M
# steps without a single success (see IMP_NOTES.md) despite the task being
# provably achievable — this seeds the replay buffer and pretrains the
# actor on real successful trajectories instead of hoping SAC randomly
# discovers one. Takes a while (single-threaded, ~150-200 steps per
# success) — only needs to run once; demonstrations.npz persists in this
# Colab session's local disk for the rest of it (re-run if the runtime
# disconnects and you start a fresh session).
!python collect_demonstrations.py 300

In [ ]:
# Cell 6 — run BC-warm-started training. This streams SB3's logging table
# live and blocks until 1,000,000 steps complete or the runtime
# disconnects — checkpoints every 12,500 steps land in Drive via the
# symlink either way, so a disconnect loses at most that much progress.
!python train_osc_grasp_bc_parallel.py

In [ ]:
# Cell 7 — only relevant if cell 6 finished without disconnecting: the FINAL
# model.save() writes to the repo directory (Colab's local disk), not
# checkpoints/ — copy it to Drive too so it isn't lost.
!cp sac_franka_osc_grasp_bc_parallel.zip /content/drive/MyDrive/Arm-OSC-Grasp-checkpoints/ 2>/dev/null || echo "Not found yet — training may not have completed."